# LangChain + LangGraph + LangSmith — Notebook Walkthrough

Same six concepts as the numbered `.py` scripts in this project, as notebook
cells instead. Run cells top to bottom, in order — later cells (agents,
LangGraph) reuse variables defined earlier (`llm`, `tools`, etc.).

**Before running anything here:** make sure this notebook is using the
project's virtual environment as its kernel, not your system Python.

If you started Jupyter from an activated venv (`source .venv/bin/activate`
then `jupyter lab`), you're already set. Otherwise, in the Jupyter menu:
**Kernel -> Change Kernel -> Python (.venv)** (or run the cell below once
to register it, then restart the kernel and pick it from that menu).

```bash
# one-time, from a terminal, inside the activated venv:
python -m ipykernel install --user --name=langchain-learning --display-name "Python (langchain-learning)"
```


## Setup — run this cell first

Loads your `.env` (Groq key, model name) so every cell below can use it.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-20b")
print("Using model:", MODEL)


## Step 1 — Talk to an LLM

Same idea as `01_llm_basics/01_hello_llm.py`: LangChain wraps every provider
(Groq here) behind the same `.invoke()` interface.

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(model=MODEL, temperature=0.2)

response = llm.invoke("In one sentence, what is LangChain used for?")
print(response.content)


In [ ]:
# In a notebook you can poke at the object interactively — try this in its own cell:
response


## Step 2 — Prompt templates and chains (LCEL)

`prompt | model | output_parser` — the `|` pipe chains pieces together.
This is the same pattern as `02_prompts_chains/02_prompt_chain.py`.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a {role}. Answer in 2 short sentences, no fluff."),
        ("human", "Explain {topic} to me."),
    ]
)

chain = prompt | llm | StrOutputParser()

print(chain.invoke({"role": "senior ML engineer", "topic": "vector embeddings"}))


In [ ]:
# Same chain, different inputs — try changing role/topic and re-running this cell.
print(chain.invoke({"role": "5-year-old", "topic": "vector embeddings"}))


## Step 3 — Tools and `create_agent`

The model decides, per turn, whether to call a tool or answer directly.
Matches `03_tools_agent/03_tool_agent.py`.

In [ ]:
from langchain.agents import create_agent

def get_weather(city: str) -> str:
    """Get the current weather for a given city."""
    fake_data = {"hyderabad": "32°C, humid", "bengaluru": "24°C, cloudy"}
    return fake_data.get(city.lower(), f"No data for {city}, assume 28°C and sunny.")

def add_numbers(a: float, b: float) -> float:
    """Add two numbers together and return the result."""
    return a + b

agent = create_agent(
    model=ChatGroq(model=MODEL, temperature=0),
    tools=[get_weather, add_numbers],
    system_prompt="You are a helpful assistant. Use the available tools when they would give a more accurate answer than guessing.",
)

result = agent.invoke({"messages": [{"role": "user", "content": "What's the weather in Hyderabad, and what's 42 + 58?"}]})
print(result["messages"][-1].content)


In [ ]:
# Inspect every step the agent took — useful for seeing exactly when/how it called a tool.
for m in result["messages"]:
    print(f"[{m.type}]", m.content if m.content else m.tool_calls)


## Step 4 — Build the agent loop yourself with LangGraph

`create_agent` above hid the mechanics. Here we build the same
START -> agent -> (tools? loop back : END) graph explicitly, matching
`04_langgraph_agent/04_langgraph_agent.py`.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

tools = [get_weather]
graph_llm = ChatGroq(model=MODEL, temperature=0).bind_tools(tools)

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

def agent_node(state: AgentState):
    response = graph_llm.invoke(state["messages"])
    return {"messages": [response]}

graph_builder = StateGraph(AgentState)
graph_builder.add_node("agent", agent_node)
graph_builder.add_node("tools", ToolNode(tools))
graph_builder.add_edge(START, "agent")
graph_builder.add_conditional_edges("agent", tools_condition, {"tools": "tools", "__end__": END})
graph_builder.add_edge("tools", "agent")
graph = graph_builder.compile()


In [ ]:
# Notebooks can render the graph as an actual image (not just ASCII) — nice for a portfolio screenshot.
from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))


In [ ]:
result = graph.invoke({"messages": [{"role": "user", "content": "What's the weather in Bengaluru?"}]})
for m in result["messages"]:
    print(f"[{m.type}]", m.content if m.content else getattr(m, "tool_calls", ""))


## Step 5 — LangSmith tracing

Nothing to change in code — LangSmith hooks in via environment variables.
Set these in `.env` (or directly in this cell for a quick test) and rerun
any cell above; then check https://smith.langchain.com for the trace.

```python
import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = "lsv2_pt_..."
os.environ["LANGSMITH_PROJECT"] = "langchain-learning"
```

One notebook-specific gotcha: environment variables set with `load_dotenv()`
at the top of this notebook are only read once. If you add `LANGSMITH_*` to
`.env` after already running the setup cell, either re-run that cell or set
`os.environ[...]` directly as above, then restart the kernel to be safe.

## Step 6 — RAG capstone

Retrieval as just another tool. This cell mirrors `06_rag_agent/06_rag_agent.py`.
The first run downloads a small (~90MB) local embedding model from Hugging Face.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader

DOCS_PATH = "../06_rag_agent/docs/servicenow_notes.txt"  # adjust if you move this notebook

loader = TextLoader(DOCS_PATH)
documents = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(chunks, embeddings, persist_directory="./.chroma_db_notebook")
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

def search_notes(query: str) -> str:
    """Search the local notes for information relevant to the query."""
    docs = retriever.invoke(query)
    return "\n---\n".join(d.page_content for d in docs) if docs else "No relevant notes found."

rag_tools = [search_notes]
rag_llm = ChatGroq(model=MODEL, temperature=0).bind_tools(rag_tools)

class RagState(TypedDict):
    messages: Annotated[list, add_messages]

def rag_agent_node(state: RagState):
    return {"messages": [rag_llm.invoke(state["messages"])]}

rag_graph_builder = StateGraph(RagState)
rag_graph_builder.add_node("agent", rag_agent_node)
rag_graph_builder.add_node("tools", ToolNode(rag_tools))
rag_graph_builder.add_edge(START, "agent")
rag_graph_builder.add_conditional_edges("agent", tools_condition, {"tools": "tools", "__end__": END})
rag_graph_builder.add_edge("tools", "agent")
rag_graph = rag_graph_builder.compile()

result = rag_graph.invoke({"messages": [{"role": "user", "content": "What does AICT do, and how does it relate to the CMDB?"}]})
print(result["messages"][-1].content)


## Where to go from here

Try changing a tool's docstring and see how it changes the model's behavior,
or add a `print(state)` inside `agent_node` to watch the state grow on each
loop iteration — that kind of quick experiment is exactly what notebooks are
good for. Once you want the polished, reviewable version of something you
built here, move it back into a `.py` file (like the numbered scripts) —
that's the better format for anything you'll commit to GitHub or reuse.